In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt


In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image

from torchvision.transforms import functional as TF

class SUIMDataset(Dataset):
    def __init__(self, img_dir, mask_dir, transform=None, size=(256,256)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.size = size

        imgs = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(".jpg")])
        masks = sorted([f for f in os.listdir(mask_dir) if f.lower().endswith((".png", ".jpg"))])

        mask_map = {}
        for m in masks:
            stem = os.path.splitext(m)[0]
            mask_map[stem] = m

        self.pairs = []
        for im in imgs:
            stem = os.path.splitext(im)[0]
            if stem in mask_map:
                self.pairs.append((im, mask_map[stem]))

        print("num matched pairs:", len(self.pairs))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.pairs[idx]

        img_path = os.path.join(self.img_dir, img_name)
        mask_path = os.path.join(self.mask_dir, mask_name)

        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        # resize BOTH to same size
        img = TF.resize(img, self.size)
        mask = TF.resize(mask, self.size, interpolation=Image.NEAREST)

        img = TF.to_tensor(img)

        mask = np.array(mask)
        mask = torch.from_numpy(mask).long()
        mask = remap_mask(mask)

        return img, mask


In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

In [ ]:
img_dir = os.path.join(path, "dataset", "images")
mask_dir = os.path.join(path, "dataset", "masks")

print("img_dir:", img_dir)
print("mask_dir:", mask_dir)

print("num images:", len(os.listdir(img_dir)))
print("num masks:", len(os.listdir(mask_dir)))


In [ ]:
dataset = SUIMDataset(img_dir, mask_dir, transform=transform)

train_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0
)

print("num samples:", len(dataset))


In [ ]:
imgs, masks = next(iter(train_loader))

plt.figure(figsize=(8,4))
for i in range(2):
    plt.subplot(2,2,i*2+1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(2,2,i*2+2)
    plt.imshow(masks[i], cmap="tab20")
    plt.title("Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
# TO DO
import torch
import segmentation_models_pytorch as smp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=8
)

model = model.to(device)
print(model.__class__.__name__)


In [ ]:
# TO DO
import torch

def train_one_epoch(model, loader, loss_fn, opt, device):
    model.train()

    total_loss = 0
    total = 0

    for imgs, masks in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)  # (B,H,W) long

        opt.zero_grad()
        out = model(imgs)         # (B,8,H,W)
        loss = loss_fn(out, masks)
        loss.backward()
        opt.step()

        total_loss += loss.item() * imgs.size(0)
        total += imgs.size(0)

    return total_loss / total

def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    total = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs = imgs.to(device)
            masks = masks.to(device)

            out = model(imgs)
            loss = loss_fn(out, masks)

            total_loss += loss.item() * imgs.size(0)
            total += imgs.size(0)

    return total_loss / total


In [ ]:
# TO DO
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print("device:", device)

loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5
train_losses = []
val_losses = []

for epoch in range(epochs):
    tr_loss = train_one_epoch(model, train_loader, loss_fn, opt, device)
    va_loss = validate_one_epoch(model, train_loader, loss_fn, device)  # same loader if only one

    train_losses.append(tr_loss)
    val_losses.append(va_loss)

    print(f"Epoch {epoch+1}/{epochs} | train loss {tr_loss:.4f} | val loss {va_loss:.4f}")


In [ ]:
plt.figure()
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()


In [ ]:
# TO DO
import numpy as np
import matplotlib.pyplot as plt
import torch

model.eval()

imgs, masks = next(iter(train_loader))
imgs = imgs.to(device)

with torch.no_grad():
    out = model(imgs)              # (B,8,H,W)
    preds = out.argmax(1).cpu()    # (B,H,W)

imgs = imgs.cpu()
masks = masks.cpu()

n = min(3, imgs.size(0))

plt.figure(figsize=(12, 4*n))
for i in range(n):
    plt.subplot(n, 3, i*3 + 1)
    plt.imshow(imgs[i].permute(1,2,0))
    plt.title("Image")
    plt.axis("off")

    plt.subplot(n, 3, i*3 + 2)
    plt.imshow(masks[i], cmap="tab20", vmin=0, vmax=7)
    plt.title("GT Mask")
    plt.axis("off")

    plt.subplot(n, 3, i*3 + 3)
    plt.imshow(preds[i], cmap="tab20", vmin=0, vmax=7)
    plt.title("Pred Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()
